In [ ]:
# Load the inventory dataset
import pandas as pd

inventory_data = pd.read_csv("../data/cleaned_dataset.csv")

inventory_data.columns.tolist()

['date',
 'site_id',
 'cement_type',
 'planned_pour_tonnes',
 'consumed_tonnes',
 'opening_inventory_tonnes',
 'deliveries_tonnes',
 'closing_inventory_tonnes',
 'rain_mm',
 'avg_temp_c',
 'silo_capacity',
 'region',
 'behavior',
 'opening_capacity_violation',
 'closing_capacity_violation']

In [5]:
#Load the saved Random Forest forecast
forecast_data = pd.read_csv(
    "../outputs/random_forest_forecasts.csv"
)

forecast_data.head()

,date,site_id,consumed_tonnes,predicted_tonnes
0,2024-11-06,SITE_001,31.27,34.8780
1,2024-11-06,SITE_002,13.89,13.8906
2,2024-11-06,SITE_003,49.50,39.9107
3,2024-11-06,SITE_004,14.56,14.5608
4,2024-11-06,SITE_005,36.09,39.7832


Table containing the predicted consumption, starting inventory, deliveries and silo capacity

In [6]:
# Convert both date columns to proper dates
inventory_data["date"] = pd.to_datetime(inventory_data["date"])
forecast_data["date"] = pd.to_datetime(forecast_data["date"])

# Combine forecast and inventory information
inventory_projection = forecast_data.merge(
    inventory_data[
        [
            "date",
            "site_id",
            "cement_type",
            "opening_inventory_tonnes",
            "deliveries_tonnes",
            "closing_inventory_tonnes",
            "silo_capacity"
        ]
    ],
    on=["date", "site_id"]
)

inventory_projection.head()

,date,site_id,consumed_tonnes,predicted_tonnes,cement_type,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,silo_capacity
0,2024-11-06,SITE_001,31.27,34.8780,CEM_III,6.10,25.17,0.00,448
1,2024-11-06,SITE_002,13.89,13.8906,CEM_I,19068.70,12.22,19067.03,288
2,2024-11-06,SITE_003,49.50,39.9107,CEM_I,185.00,33.33,168.83,314
3,2024-11-06,SITE_004,14.56,14.5608,CEM_III,19453.41,18.81,19457.66,472
4,2024-11-06,SITE_005,36.09,39.7832,CEM_III,0.00,36.09,0.00,230


In [7]:
#Making some projection

# Arrange every site in date order
inventory_projection = inventory_projection.sort_values(
    ["site_id", "date"]
)

# Calculate daily inventory change
inventory_projection["inventory_change"] = (
    inventory_projection["deliveries_tonnes"]
    - inventory_projection["predicted_tonnes"]
)

# Get the starting inventory for each site
inventory_projection["starting_inventory"] = (
    inventory_projection.groupby("site_id")[
        "opening_inventory_tonnes"
    ].transform("first")
)

# Calculate inventory for each day
inventory_projection["inventory_tonnes"] = (
    inventory_projection["starting_inventory"]
    + inventory_projection.groupby("site_id")[
        "inventory_change"
    ].cumsum()
)

# Display the result
inventory_projection[
    [
        "date",
        "site_id",
        "starting_inventory",
        "deliveries_tonnes",
        "predicted_tonnes",
        "inventory_tonnes"
    ]
].head(10)

,date,site_id,starting_inventory,deliveries_tonnes,predicted_tonnes,inventory_tonnes
0,2024-11-06,SITE_001,6.1,25.17,34.8780,-3.6080
30,2024-11-07,SITE_001,6.1,41.23,23.2376,14.3844
60,2024-11-08,SITE_001,6.1,48.88,30.8429,32.4215
90,2024-11-09,SITE_001,6.1,36.63,35.5245,33.5270
120,2024-11-10,SITE_001,6.1,16.67,37.5600,12.6370
150,2024-11-11,SITE_001,6.1,16.96,23.0634,6.5336
180,2024-11-12,SITE_001,6.1,36.92,28.0784,15.3752
210,2024-11-13,SITE_001,6.1,17.62,0.0000,32.9952
240,2024-11-14,SITE_001,6.1,36.58,37.2863,32.2889
270,2024-11-15,SITE_001,6.1,20.23,37.3637,15.1552


In [9]:
# Saving it as dataset
inventory_projection.to_csv(
    "../outputs/inventory_forecast.csv",
    index=False
)